# Implement Contrastive Loss (InfoNCE) + CLIP Training Loop

**Difficulty**: 🟡 Medium

**Companies**: OpenAI, Anthropic, DeepMind, Midjourney, Apple

---

### Problem Statement

CLIP (Contrastive Language–Image Pre-training) learns to align image and text representations in a shared embedding space using **InfoNCE contrastive loss**. Given a batch of N (image, text) pairs, the model is trained so that the N correct pairings have high cosine similarity while the N²−N incorrect pairings have low similarity.

Your task is to implement:
1. The **InfoNCE contrastive loss** — a symmetric cross-entropy loss over the cosine similarity matrix with a learnable temperature parameter.
2. A **SimpleCLIP** model with separate image and text encoders (simple linear projections) and a learnable log-temperature.
3. A minimal training loop on synthetic paired data that demonstrates the loss decreasing and the similarity matrix becoming more diagonal.

---

### Requirements

1. **`info_nce_loss(image_features, text_features, temperature)`** — Compute cosine similarity matrix, then symmetric cross-entropy (average of image→text and text→image cross-entropy losses).
2. **`SimpleCLIP(nn.Module)`** — Contains `image_encoder`, `text_encoder` (both `nn.Linear`), and a learnable `temperature` parameter (stored as log-temperature for numerical stability).
3. **Training** — Loss should decrease over steps. The similarity matrix should become increasingly diagonal.

---

### Constraints

- ✅ Features must be L2-normalized before computing cosine similarity.
- ✅ Temperature is learnable and applied as a scaling factor to the similarity matrix.
- ✅ The loss is symmetric: average of (image→text CE) and (text→image CE).
- ❌ Do **not** use any pre-trained models or external data.

---

<details>
  <summary>💡 Hint</summary>

  - Cosine similarity: normalize both feature sets with `F.normalize`, then matrix-multiply: `image_features @ text_features.T`.
  - The labels for cross-entropy are simply `torch.arange(batch_size)` — each image i should match text i.
  - Store temperature as `nn.Parameter(torch.log(torch.tensor(1/0.07)))` and use `self.temperature.exp()` when scaling.
  - The symmetric loss = `(CE(logits, labels) + CE(logits.T, labels)) / 2`.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Synthetic paired data
torch.manual_seed(42)

batch_size = 32
image_dim = 128   # raw image feature dimension
text_dim = 64     # raw text feature dimension
embed_dim = 64    # shared embedding dimension

# Create synthetic "paired" data: each image_i corresponds to text_i
# We create a shared latent factor so correct pairs are learnable
latent = torch.randn(batch_size, embed_dim)
images = torch.randn(batch_size, image_dim) + latent @ torch.randn(embed_dim, image_dim) * 0.1
texts = torch.randn(batch_size, text_dim) + latent @ torch.randn(embed_dim, text_dim) * 0.1

print(f"Images shape: {images.shape}")
print(f"Texts shape:  {texts.shape}")

In [ ]:
def info_nce_loss(image_features: torch.Tensor, text_features: torch.Tensor,
                  temperature: torch.Tensor) -> torch.Tensor:
    """
    Compute the InfoNCE (CLIP) contrastive loss.

    Args:
        image_features: (batch_size, embed_dim) - already projected, NOT yet normalized
        text_features:  (batch_size, embed_dim) - already projected, NOT yet normalized
        temperature:    scalar tensor (the exponentiated log-temperature)

    Returns:
        Scalar loss value (symmetric cross-entropy).
    """
    # TODO: Step 1 - L2-normalize both feature sets along the embedding dimension

    # TODO: Step 2 - Compute the cosine similarity matrix scaled by temperature
    #                 logits shape should be (batch_size, batch_size)

    # TODO: Step 3 - Create labels: the diagonal entries are the correct pairs
    #                 labels = [0, 1, 2, ..., batch_size-1]

    # TODO: Step 4 - Compute symmetric cross-entropy loss
    #                 loss_i2t = cross_entropy(logits, labels)      # image-to-text
    #                 loss_t2i = cross_entropy(logits.T, labels)    # text-to-image
    #                 return average of both

    ...


class SimpleCLIP(nn.Module):
    def __init__(self, image_dim: int, text_dim: int, embed_dim: int):
        """
        A minimal CLIP-style model.

        Args:
            image_dim: dimension of raw image features
            text_dim:  dimension of raw text features
            embed_dim: shared embedding space dimension
        """
        super().__init__()
        # TODO: Create image_encoder (nn.Linear: image_dim -> embed_dim)

        # TODO: Create text_encoder (nn.Linear: text_dim -> embed_dim)

        # TODO: Create learnable log-temperature parameter
        #       Initialize so that temperature = 1/0.07 ~ 14.3
        #       Use nn.Parameter(torch.log(torch.tensor(1.0 / 0.07)))

        ...

    def forward(self, images: torch.Tensor, texts: torch.Tensor):
        """
        Forward pass.

        Returns:
            loss: scalar InfoNCE loss
            image_features: projected image embeddings
            text_features: projected text embeddings
        """
        # TODO: Project images and texts through their encoders

        # TODO: Compute temperature from log_temperature

        # TODO: Compute and return (loss, image_features, text_features)

        ...


# Quick test
model = SimpleCLIP(image_dim, text_dim, embed_dim)
loss, img_feat, txt_feat = model(images, texts)
print(f"Initial loss: {loss.item():.4f}")
print(f"Image features shape: {img_feat.shape}")
print(f"Text features shape:  {txt_feat.shape}")

In [ ]:
# Training loop and validation
model = SimpleCLIP(image_dim, text_dim, embed_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

losses = []
num_steps = 200

for step in range(num_steps):
    optimizer.zero_grad()
    loss, img_feat, txt_feat = model(images, texts)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step % 50 == 0:
        print(f"Step {step:3d} | Loss: {loss.item():.4f} | Temp: {model.log_temperature.exp().item():.2f}")

print(f"\nFinal loss: {losses[-1]:.4f}")

# Validation 1: Loss should decrease
assert losses[-1] < losses[0], "Loss did not decrease during training!"
print("PASSED: Loss decreased.")

# Validation 2: Similarity matrix should be more diagonal
with torch.no_grad():
    _, img_feat, txt_feat = model(images, texts)
    img_norm = F.normalize(img_feat, dim=-1)
    txt_norm = F.normalize(txt_feat, dim=-1)
    sim_matrix = img_norm @ txt_norm.T

    # Diagonal mean should be higher than off-diagonal mean
    diag_mean = sim_matrix.diag().mean().item()
    mask = ~torch.eye(batch_size, dtype=torch.bool)
    offdiag_mean = sim_matrix[mask].mean().item()
    print(f"Diagonal similarity mean:     {diag_mean:.4f}")
    print(f"Off-diagonal similarity mean: {offdiag_mean:.4f}")
    assert diag_mean > offdiag_mean, "Diagonal similarity should be higher than off-diagonal!"
    print("PASSED: Similarity matrix is more diagonal.")

print("\nAll tests passed!")